# Baseline: standard LFM3 LoRA fine-tune (JSON output)

The control for Decide-then-Write: LFM3-1.7B fine-tuned the normal way, generating the full JSON answer, with `ml/finetune/train_lora.py` and `ml/finetune/config-lfm3.yaml` on the same 8000 generated examples rendered with the production prompt (v2, compact). Exports Q4_K_M / Q8_0 GGUF with `ml/finetune/export_gguf.sh`.

In [ ]:
# Setup: clone the experiment branch and install dependencies
import os, subprocess
REPO = "github.com/AwaisSDev/Codeply-Craft"
BRANCH = "codeply-li-dtw-experiment"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@{REPO}"
except Exception:
    url = f"https://{REPO}"
if not os.path.exists("/kaggle/working/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/codeply"], check=True)
os.chdir("/kaggle/working/codeply/research/dtw")
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece", shell=True, check=True)
subprocess.run("cd data/out && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp at the pinned commit: GGUF converter and quantizer
import os, subprocess
if not os.path.exists("/kaggle/working/llama.cpp"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp && cd /kaggle/working/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    subprocess.run("cd /kaggle/working/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize > /dev/null", shell=True, check=True)
    subprocess.run("pip -q install -e /kaggle/working/llama.cpp/gguf-py", shell=True, check=True)

def export_gguf(run):
    """backbone -> F16 GGUF -> Q8_0 and Q4_0; heads -> heads.bin/json."""
    subprocess.run(f"python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {run}/backbone --outtype f16 --outfile {run}/backbone-f16.gguf", shell=True, check=True)
    for q in ("Q8_0", "Q4_0"):
        subprocess.run(f"/kaggle/working/llama.cpp/build/bin/llama-quantize {run}/backbone-f16.gguf {run}/backbone-{q.lower()}.gguf {q}", shell=True, check=True)
    subprocess.run(f"python model/export_heads.py {run}", shell=True, check=True)

In [ ]:
import subprocess, os
os.chdir("/kaggle/working/codeply")
subprocess.run("pip -q install -r ml/finetune/requirements.txt", shell=True, check=True)
subprocess.run("cp research/dtw/data/out/sft-train.jsonl research/dtw/data/out/sft-val.jsonl ml/data/out/", shell=True, check=True)
subprocess.run("python ml/finetune/train_lora.py --config ml/finetune/config-lfm3.yaml", shell=True, check=True)

In [ ]:
!cd /kaggle/working/llama.cpp && cmake --build build -j4 --target llama-imatrix > /dev/null
!cd /kaggle/working/codeply && LLAMA_CPP=/kaggle/working/llama.cpp bash ml/finetune/export_gguf.sh ml/finetune/runs/codeply-context-v1-lfm3/merged codeply-context-v1-lfm3
!cd /kaggle/working/codeply/ml/finetune/runs && zip -qr /kaggle/working/lfm3-lora-baseline.zip . -i "*.gguf" "*train-log.jsonl"
!ls -la /kaggle/working/lfm3-lora-baseline.zip